# Logging

PyLabRobot uses standard Python [`logging`](https://docs.python.org/3/library/logging.html). All loggers live under the `"pylabrobot"` name, so the usual Python techniques for setting levels, adding handlers, and filtering by module all work exactly as you'd expect.

## Log levels

Python's five built-in levels apply as normal:

| Level | Value | Typical content |
|-------|-------|-----------------|
| `IO` | 5 | Raw bytes sent to / received from hardware |
| `DEBUG` | 10 | Internal state, retry logic, per-well progress |
| `INFO` | 20 | High-level actions: setup, teardown, measurement starts |
| `WARNING` | 30 | Recoverable problems (timeouts, fallbacks) |
| `ERROR` | 40 | Failures that stop an operation |
| `CRITICAL` | 50 | Unrecoverable errors |

PyLabRobot adds one extra level: **`IO`** (value 5), which sits below `DEBUG`. It exists because hardware backends can generate a large volume of raw hex dumps (every USB/serial frame sent and received). Mixing those into `DEBUG` would make debug logs unreadable during normal development, so they are separated into their own level that you only enable when you are actively inspecting wire traffic.

## Quick start

In Jupyter notebooks, PLR automatically enables console logging at `INFO` level. In scripts you need to opt in.

In [ ]:
import pylabrobot

# Enable console output at INFO level (the default)
pylabrobot.verbose(True)

To see `DEBUG` messages:

In [ ]:
import logging

pylabrobot.verbose(True, level=logging.DEBUG)

To see everything including raw IO bytes:

In [ ]:
from pylabrobot.io import LOG_LEVEL_IO

pylabrobot.verbose(True, level=LOG_LEVEL_IO)

To turn console output off again:

In [ ]:
pylabrobot.verbose(False)

## Logging to a file

Use `setup_logger` to write logs to disk. This is useful for long-running protocols where you want a persistent record.

In [ ]:
pylabrobot.setup_logger(log_dir="./logs", level=logging.DEBUG)

This creates a file like `logs/pylabrobot-20260312.log` (dated by day). The file handler captures everything at or above the level you set. You can combine this with `verbose()` to also see output in the console.

Pass `log_dir=None` to disable file logging.

## Config file

Instead of calling `setup_logger` in every script, you can create a `pylabrobot.ini` (or `pylabrobot.json`) in your project directory:

```ini
[logging]
level = DEBUG
log_dir = ./logs
```

```json
{
  "logging": {
    "level": "DEBUG",
    "log_dir": "./logs"
  }
}
```

PLR searches for this file in the current directory and all parent directories. The config is loaded automatically on `import pylabrobot`.

## Filtering by module

Since PLR uses Python's logger hierarchy, you can adjust the level for a specific backend without changing the global level:

In [ ]:
import logging

# Only show warnings and above from the STAR backend
logging.getLogger("pylabrobot.liquid_handling.backends.hamilton.STAR").setLevel(logging.WARNING)

# Show debug output from the plate reader
logging.getLogger("pylabrobot.plate_reading").setLevel(logging.DEBUG)

## Summary

| Goal | Code |
|------|------|
| Console output (INFO) | `pylabrobot.verbose(True)` |
| Console output (DEBUG) | `pylabrobot.verbose(True, level=logging.DEBUG)` |
| Console output (IO) | `pylabrobot.verbose(True, level=LOG_LEVEL_IO)` |
| Log to file | `pylabrobot.setup_logger(log_dir="./logs", level=logging.DEBUG)` |
| Config file | `pylabrobot.ini` or `pylabrobot.json` in project root |
| Filter one module | `logging.getLogger("pylabrobot.module").setLevel(...)` |
| Disable console | `pylabrobot.verbose(False)` |